# 01 · Exploring delays in Baden-Württemberg

Piebro monthly data filtered to the ~1,000 BW rail stations matched to the GTFS feed (`ml/pipelines/download_data.py`).

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(ROOT / "backend"))
import numpy as np, pandas as pd
pd.set_option("display.width", 140)

In [2]:
from app.engine.features import events_from_stops
stops = pd.read_parquet(ROOT / "data/processed/delays-2026-08.parquet")
events = events_from_stops(stops[~stops.is_additional_stop])
ok = events[~events.cancelled & events.delay.notna()]
print(f"{len(stops):,} stop rows, {len(events):,} arrival/departure events in August 2026")
ok.delay.describe(percentiles=[.5, .8, .9, .95, .99]).round(2)

2,988,243 stop rows, 5,296,217 arrival/departure events in August 2026


count    5118725.00
mean           3.60
std            8.32
min        -1438.00
50%            1.00
80%            5.00
90%            9.00
95%           16.00
99%           38.00
max          512.00
Name: delay, dtype: float64

## Delay by product and hour
Long-distance trains arrive with much heavier tails; the evening peak is the worst time of day for regional trains.

In [3]:
by_product = ok.groupby("product").delay.agg(
    events="size", median="median", p90=lambda s: s.quantile(.9), share_over_5=lambda s: (s > 5).mean())
by_product.round(2)

,events,median,p90,share_over_5
product,,,,
FV,204783,5.0,38.0,0.47
RB,1589155,1.0,9.0,0.18
RE,1019564,2.0,14.0,0.27
S,2305223,1.0,6.0,0.10


In [4]:
rb_re = ok[ok["product"].isin(["RE", "RB"]) & (ok.is_departure == 0)]
rb_re.groupby("hour").delay.agg(p50="median", p90=lambda s: s.quantile(.9)).T.round(1)

hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
p50,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,...,2.0,2.0,2.0,2.0,2.0,1.0,1.0,2.0,2.0,1.0
p90,12.0,11.0,10.0,6.3,5.0,6.0,7.0,9.0,10.0,10.0,...,12.0,12.0,12.0,13.0,13.0,13.0,12.0,13.0,13.0,12.0


## Cancellations by line (departures, lines with ≥ 2,000 departures)

In [5]:
dep = events[events.is_departure == 1]
lines = dep.groupby("line").cancelled.agg(departures="size", rate="mean")
lines[lines.departures >= 2000].sort_values("rate", ascending=False).head(12).round(3)

,departures,rate
line,,
90,4002,0.339
RE90,9020,0.277
RB12,2596,0.247
RE89,12197,0.221
RB89,13130,0.211
RS2,3476,0.181
RB90,3498,0.168
RE13,2968,0.164
RS21,10514,0.152
